# Aula NN — 01. Introdução

> Pré-requisito: `.env` da raiz configurado com a sua instância de SQL Server e kernel **Python (Estudo-MDS)** selecionado.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore", category=UserWarning)   # aviso do pandas sobre conexão pyodbc

# O kernel abre na pasta notebooks/; a raiz da aula é a pasta de cima.
AULA = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.append(str(AULA))   # permite "from src import ..."

from mds_comum import sqlserver
from src import config

## Criando o banco da aula

Cada aula tem o seu banco dentro da instância compartilhada. `garantir_banco` só cria se ainda não existir.

In [ ]:
import pandas as pd

sqlserver.garantir_banco(config.SQL_DATABASE)

with sqlserver.conectar(config.SQL_DATABASE) as conn:
    display(pd.read_sql("SELECT DB_NAME() AS banco, @@VERSION AS versao", conn))

## Criando os objetos: sempre `DROP ... IF EXISTS` antes

Os notebooks das aulas são feitos para rodar **quantas vezes for preciso**: cada objeto é
apagado antes de ser criado de novo. Assim dá para refazer a demonstração ao vivo sem erro
de "já existe" — e sem `IF ... IS NULL`, que esconderia a criação, que é justamente o que
queremos mostrar.

A ordem do `DROP` é a inversa da criação: primeiro quem depende (procedures, views, tabelas
filhas), depois quem é dependido (tabelas mãe, tipos, schema).

In [ ]:
with sqlserver.conectar(config.SQL_DATABASE, autocommit=True) as conn:
    sqlserver.executar_script(conn, '''
    DROP TABLE  IF EXISTS aula.Exemplo;
    DROP SCHEMA IF EXISTS aula;
    GO
    CREATE SCHEMA aula;
    GO
    CREATE TABLE aula.Exemplo (
        ExemploId INT IDENTITY(1,1) CONSTRAINT PK_Exemplo PRIMARY KEY,
        Nome      NVARCHAR(100) NOT NULL
    );
    ''')
    display(pd.read_sql("""
        SELECT s.name AS [schema], o.name AS objeto, o.type_desc AS tipo
        FROM sys.objects o JOIN sys.schemas s ON s.schema_id = o.schema_id
        WHERE s.name = 'aula'
    """, conn))